# IND320, Assignment 3
## By: Ole Andreas Grøtting
Github: [IND320 - OLEGRO99](https://github.com/olegro99/IND320-OLEGRO99.git)\
Streamlit: https://ind320-olegro99-ybwpxf3ssrwyekrthttcuc.streamlit.app/

## imports

In [20]:
import pandas as pd
import numpy as np
import requests
from entsoe import EntsoePandasClient
import os
from dotenv import load_dotenv

# downloading data

In [2]:
def download_exchange_rates(start_date, end_date):

    url = (
        "https://data.norges-bank.no/api/data/EXR/"
        "B.EUR+USD.NOK.SP"
        f"?format=csv&startPeriod={start_date}"
        f"&endPeriod={end_date}"
        "&locale=en&bom=include"
    )

    # Download data from Norges Bank
    df = pd.read_csv(url, sep=";")

    return df

In [3]:
exchange_rates_df = download_exchange_rates(
    start_date="2023-01-01", 
    end_date="2023-12-31"
    )

exchange_rates_df.head()

,FREQ,Frequency,BASE_CUR,Base Currency,QUOTE_CUR,Quote Currency,TENOR,Tenor,DECIMALS,CALCULATED,UNIT_MULT,Unit Multiplier,COLLECTION,Collection Indicator,TIME_PERIOD,OBS_VALUE
0,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2023-01-02,9.8413
1,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2023-01-03,9.9839
2,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2023-01-04,10.1311
3,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2023-01-05,10.1168
4,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2023-01-06,10.2924


## checking time zones

In [8]:
# Download NVE data
base_url = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData"

response = requests.get(base_url)
response.raise_for_status()

data = response.json()
df = pd.DataFrame(data)

# Remove invalid dates
df = df[df["neste_Publiseringsdato"] != "0001-01-01T00:00:00"]

# Convert date column
df["dato_Id"] = pd.to_datetime(df["dato_Id"])

print(df["dato_Id"].head())
print(df["dato_Id"].dtype)
print(df["dato_Id"].dt.tz)

0    2020-11-15
7    2023-08-13
10   2021-04-18
11   2019-06-02
13   2019-10-27
Name: dato_Id, dtype: datetime64[us]
datetime64[us]
None


In [11]:
# Checking timezone for Norges Bank
nb_dates = pd.to_datetime(exchange_rates_df["TIME_PERIOD"])

print(nb_dates.head())
print(nb_dates.dtype)
print(nb_dates.dt.tz)

# Check whether timestamps contain a time component
print(nb_dates.dt.strftime("%Y-%m-%d %H:%M:%S").head())

0   2023-01-02
1   2023-01-03
2   2023-01-04
3   2023-01-05
4   2023-01-06
Name: TIME_PERIOD, dtype: datetime64[us]
datetime64[us]
None
0    2023-01-02 00:00:00
1    2023-01-03 00:00:00
2    2023-01-04 00:00:00
3    2023-01-05 00:00:00
4    2023-01-06 00:00:00
Name: TIME_PERIOD, dtype: str


In [24]:
# Connect to the ENTSO-E API
client = EntsoePandasClient(
    api_key=os.getenv("ENTSOE_API_KEY")
)

# Define the two-year period in UTC
start = pd.Timestamp("2024-10-15", tz="UTC")
end = pd.Timestamp("2026-10-15", tz="UTC")

# Query physical cross-border electricity flows
flows = client.query_crossborder_flows(
    country_code_from="NO_2",
    country_code_to="DK_1",
    start=start,
    end=end
)

# Convert the returned time series into a DataFrame
flows_df = flows.rename("flow_mw").reset_index()

# Display the retrieved data
print(flows_df.head())
print(flows_df.shape)

                      index  flow_mw
0 2024-10-15 02:00:00+02:00  1386.84
1 2024-10-15 03:00:00+02:00  1386.69
2 2024-10-15 04:00:00+02:00  1386.92
3 2024-10-15 05:00:00+02:00  1387.20
4 2024-10-15 06:00:00+02:00  1387.27
(57480, 2)


In [25]:
# Check timezone for ENTSO-E data

print(flows_df.head())
print(flows_df.dtypes)

# Convert timestamp column to datetime
flows_df["index"] = pd.to_datetime(flows_df["index"])

# Check timezone
print("Timezone:", flows_df["index"].dt.tz)

# Display timestamps
print(
    flows_df["index"]
    .head(10)
    .dt.strftime("%Y-%m-%d %H:%M:%S %Z")
)

                      index  flow_mw
0 2024-10-15 02:00:00+02:00  1386.84
1 2024-10-15 03:00:00+02:00  1386.69
2 2024-10-15 04:00:00+02:00  1386.92
3 2024-10-15 05:00:00+02:00  1387.20
4 2024-10-15 06:00:00+02:00  1387.27
index      datetime64[us, Europe/Oslo]
flow_mw                        float64
dtype: object
Timezone: Europe/Oslo
0    2024-10-15 02:00:00 CEST
1    2024-10-15 03:00:00 CEST
2    2024-10-15 04:00:00 CEST
3    2024-10-15 05:00:00 CEST
4    2024-10-15 06:00:00 CEST
5    2024-10-15 07:00:00 CEST
6    2024-10-15 08:00:00 CEST
7    2024-10-15 09:00:00 CEST
8    2024-10-15 10:00:00 CEST
9    2024-10-15 11:00:00 CEST
Name: index, dtype: str


In [26]:
#timezones open meteo
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 59.66,       # Example: Ås
    "longitude": 10.78,
    "start_date": "2023-01-01",
    "end_date": "2023-12-31",
    "hourly": [
        "temperature_2m",
        "precipitation",
        "relative_humidity_2m",
        "wind_speed_10m"
    ],
    "timezone": "GMT"
}

response = requests.get(url, params=params)
response.raise_for_status()

weather_data = response.json()

weather_df = pd.DataFrame(weather_data["hourly"])

# Convert timestamps to datetime
weather_df["time"] = pd.to_datetime(weather_df["time"])

# Check timezone
print(weather_df.head())
print(weather_df.dtypes)

print("Timezone:", weather_df["time"].dt.tz)

# Display timestamps
print(
    weather_df["time"]
    .head(10)
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)

                 time  temperature_2m  precipitation  relative_humidity_2m  \
0 2023-01-01 00:00:00            -4.3            0.0                    85   
1 2023-01-01 01:00:00            -2.5            0.0                    84   
2 2023-01-01 02:00:00            -2.6            0.0                    83   
3 2023-01-01 03:00:00            -3.2            0.0                    82   
4 2023-01-01 04:00:00            -3.1            0.0                    80   

   wind_speed_10m  
0            11.4  
1             7.2  
2             7.2  
3             6.8  
4             8.0  
time                    datetime64[us]
temperature_2m                 float64
precipitation                  float64
relative_humidity_2m             int64
wind_speed_10m                 float64
dtype: object
Timezone: None
0    2023-01-01 00:00:00
1    2023-01-01 01:00:00
2    2023-01-01 02:00:00
3    2023-01-01 03:00:00
4    2023-01-01 04:00:00
5    2023-01-01 05:00:00
6    2023-01-01 06:00:00
7    2023-01

All data sources use timezone-naive datetime64[us] timestamps, except ENTSO-E, which uses datetime64[us, Europe/Oslo]. Since I set the timezone to GMT when retrieving data from Open-Meteo, those timestamps represent UTC. Therefore, I will convert the ENTSO-E timestamps from Europe/Oslo to UTC and localize the Open-Meteo timestamps as UTC to ensure consistency between the two datasets.

In [27]:
# Convert ENTSO-E timestamps to UTC
flows_df["index"] = flows_df["index"].dt.tz_convert("UTC")

# Localize Open-Meteo timestamps as UTC
weather_df["time"] = weather_df["time"].dt.tz_localize("UTC")

In [28]:
# Display timestamps with timezone information
print(flows_df["index"].dt.strftime(
    "%Y-%m-%d %H:%M:%S %Z %z"
).head(10))

0    2024-10-15 00:00:00 UTC +0000
1    2024-10-15 01:00:00 UTC +0000
2    2024-10-15 02:00:00 UTC +0000
3    2024-10-15 03:00:00 UTC +0000
4    2024-10-15 04:00:00 UTC +0000
5    2024-10-15 05:00:00 UTC +0000
6    2024-10-15 06:00:00 UTC +0000
7    2024-10-15 07:00:00 UTC +0000
8    2024-10-15 08:00:00 UTC +0000
9    2024-10-15 09:00:00 UTC +0000
Name: index, dtype: str


In [31]:
# Check that both datasets use UTC
print("ENTSO-E timezone:", flows_df["index"].dt.tz)
print("Open-Meteo timezone:", weather_df["time"].dt.tz)

# Compare the first timestamps in each dataset
print("ENTSO-E:", flows_df["index"].head())
print("Open-Meteo:", weather_df["time"].head())

ENTSO-E timezone: UTC
Open-Meteo timezone: UTC
ENTSO-E: 0   2024-10-15 00:00:00+00:00
1   2024-10-15 01:00:00+00:00
2   2024-10-15 02:00:00+00:00
3   2024-10-15 03:00:00+00:00
4   2024-10-15 04:00:00+00:00
Name: index, dtype: datetime64[us, UTC]
Open-Meteo: 0   2023-01-01 00:00:00+00:00
1   2023-01-01 01:00:00+00:00
2   2023-01-01 02:00:00+00:00
3   2023-01-01 03:00:00+00:00
4   2023-01-01 04:00:00+00:00
Name: time, dtype: datetime64[us, UTC]


In [32]:
# Convert timestamps into sets
entsoe_times = set(flows_df["index"])
weather_times = set(weather_df["time"])

# Find timestamps that exist in both datasets
common_times = entsoe_times.intersection(weather_times)

# Display the number of observations
print("ENTSO-E observations:", len(entsoe_times))
print("Open-Meteo observations:", len(weather_times))
print("Common timestamps:", len(common_times))

ENTSO-E observations: 57480
Open-Meteo observations: 8760
Common timestamps: 0


In [33]:
#comparing all 4 timestamps
# Create an overview of timestamp information for all data sources

timezone_overview = pd.DataFrame({
    "Data source": [
        "NVE Magasinstatistikk",
        "ENTSO-E",
        "Open-Meteo",
        "Norges Bank"
    ],
    "Timestamp column": [
        "dato_Id",
        "index",
        "time",
        "TIME_PERIOD"
    ],
    "Data type": [
        str(df["dato_Id"].dtype),
        str(flows_df["index"].dtype),
        str(weather_df["time"].dtype),
        str(nb_dates.dtype)
    ],
    "Timezone": [
        str(df["dato_Id"].dt.tz),
        str(flows_df["index"].dt.tz),
        str(weather_df["time"].dt.tz),
        str(nb_dates.dt.tz)
    ]
})

timezone_overview

,Data source,Timestamp column,Data type,Timezone
0,NVE Magasinstatistikk,dato_Id,datetime64[us],None
1,ENTSO-E,index,"datetime64[us, UTC]",UTC
2,Open-Meteo,time,"datetime64[us, UTC]",UTC
3,Norges Bank,TIME_PERIOD,datetime64[us],None


In [34]:
# Define timestamp columns for each data source
date_columns = {
    "NVE": df["dato_Id"],
    "ENTSO-E": flows_df["index"],
    "Open-Meteo": weather_df["time"],
    "Norges Bank": nb_dates
}

# Display the observation period for each data source
for name, dates in date_columns.items():
    print(f"\n{name}")
    print("Start:", dates.min())
    print("End:", dates.max())
    print("Observations:", len(dates))


NVE
Start: 2019-02-17 00:00:00
End: 2026-11-08 00:00:00
Observations: 3636

ENTSO-E
Start: 2024-10-15 00:00:00+00:00
End: 2026-10-14 23:45:00+00:00
Observations: 57480

Open-Meteo
Start: 2023-01-01 00:00:00+00:00
End: 2023-12-31 23:00:00+00:00
Observations: 8760

Norges Bank
Start: 2023-01-02 00:00:00
End: 2023-12-29 00:00:00
Observations: 502


In [35]:
# NVE
print("NVE:")
print(df["dato_Id"].sort_values().diff().value_counts().head())

# ENTSO-E
print("\nENTSO-E:")
print(flows_df["index"].sort_values().diff().value_counts().head())

# Open-Meteo
print("\nOpen-Meteo:")
print(weather_df["time"].sort_values().diff().value_counts().head())

# Norges Bank
print("\nNorges Bank:")
print(nb_dates.sort_values().diff().value_counts().head())

NVE:
dato_Id
0 days    3232
7 days     403
Name: count, dtype: int64

ENTSO-E:
index
0 days 00:15:00    53279
0 days 01:00:00     4200
Name: count, dtype: int64

Open-Meteo:
time
0 days 01:00:00    8759
Name: count, dtype: int64

Norges Bank:
TIME_PERIOD
0 days    251
1 days    198
3 days     48
4 days      2
6 days      1
Name: count, dtype: int64


The four data sources have different temporal resolutions. NVE contains multiple observations per date, with weekly intervals between some dates. ENTSO-E mainly provides data at 15-minute intervals, while Open-Meteo provides hourly observations. Norges Bank provides daily exchange rates on business days, resulting in gaps during weekends and holidays.

Regarding time zones, ENTSO-E and Open-Meteo timestamps are represented in UTC. NVE and Norges Bank contain date-based observations without explicit timezone information. Therefore, the original dates were retained rather than assigning an assumed timezone. This avoids introducing unnecessary timezone conversions